# Meta-Learning Pipeline v4 — LLM-Enhanced Version

**Builds on v3 by integrating Google Gemini LLM for intelligent feature interaction suggestions**

### Key Differences from v3
1. **LLM-Enhanced ArithmeticInteractions**: Uses Gemini 2.5 Flash Lite to suggest semantically meaningful
   feature interactions based on column names and statistics, combined with correlation-based pairs.
2. **Hybrid Feature Engineering**: Two sources of interaction ideas:
   - **Gemini LLM** → Semantic suggestions (understands column names, suggests meaningful operations)
   - **Correlation Analysis** → Statistical pairs from `preprocessing_function.py`
   - Results are merged, deduplicated, and quality-filtered.
3. **LLM Caching**: API is called once per dataset (keyed by column names), cached across 5 CV folds.
4. **Graceful Fallback**: If LLM is unavailable, falls back to correlation-only (v3 behavior).
5. **All other methods unchanged**: Same 13 methods as v3 (6 pipeline-based + 7 custom).

### Method Source Map

| Method | Source | Why |
|--------|--------|-----|
| ImputeMedian | `preprocessing_pipeline._fit_imputer_step` | Has fit/transform |
| MissingIndicator | `preprocessing_pipeline._fit_indicator_step` | Has fit/transform |
| StandardScaler | `preprocessing_pipeline._fit_scaler_step` | Has fit/transform |
| RobustScaler | `preprocessing_pipeline._fit_robust_scaler_step` | Has fit/transform |
| YeoJohnson | `preprocessing_pipeline._fit_power_transform_step` | Has fit/transform |
| ClipOutliers | `preprocessing_pipeline._fit_clip_outliers_step` | Has fit/transform |
| FrequencyEncoding | Custom wrapper | `pf.frequency_encode` has no fit/transform |
| TargetEncoding | Custom wrapper | Not in existing codebase |
| OneHotEncoding | Custom (sklearn OHE) | `pf.one_hot_encode` uses `pd.get_dummies` (stateless) |
| LogTransform | Custom wrapper | Not in pipeline |
| **ArithmeticInteractions** | **Gemini LLM + `pf.create_features_from_correlation_analysis`** | **Hybrid: LLM semantic + statistical correlation** |
| FastICA | `pf.apply_fastica` internals + `_create_ica_interactions` | Hybrid mode: replace low-var cols with ICA components |
| PCA_95 | Hybrid (pipeline + sklearn) | PCA not in pipeline |

### Pipeline Overview
1. **Step 1**: Leakage-free evaluation engine + preprocessing methods (6 pipeline + 7 custom + 1 LLM-enhanced)
2. **Step 2**: Meta-feature extraction (70 features)
3. **Step 3**: Collect performance labels from OpenML CC18 (Suite 99)
4. **Step 4**: Train meta-model regressors (one per preprocessing method)
5. **Step 5**: Inference — recommend preprocessing for a new dataset

### Requirements
- `google-genai` package
- `.env` file with `GEMINI_API_KEY=your_key_here`

In [1]:
import openml
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import early_stopping
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder, OneHotEncoder as SklearnOHE, StandardScaler as SklearnStdScaler
from sklearn.decomposition import PCA, FastICA
from scipy.stats import skew, kurtosis, entropy
import joblib
import os
import time
import io
import json
import re
import contextlib
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# Import from existing preprocessing infrastructure
# ============================================================
from preprocessing_pipeline import (
    fit_preprocessing_pipeline,
    apply_fitted_pipeline,
)
from preprocessing_function import (
    create_features_from_correlation_analysis,
    _find_high_corr_pairs,
    _generate_candidate_features,
    _apply_basic_filter,
    _create_ica_interactions,
)

# ============================================================
# Gemini LLM Setup
# ============================================================
from dotenv import load_dotenv
from google import genai

load_dotenv()  # Load .env file

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    print("⚠️ GEMINI_API_KEY not found in .env file. LLM features will be disabled.")
    gemini_client = None
else:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("✅ Gemini client initialized.")

# LLM suggestion cache: avoids redundant API calls for the same dataset
_llm_suggestion_cache = {}

# ============================================================
# Configuration
# ============================================================
SUITE_ID = 99                              # OpenML-CC18
RESULTS_FILE = "performance_labels_v4.csv"
META_FEATURES_FILE = "meta_features_v4.csv"
MODEL_DIR = "meta_models_v4"
MAX_ROWS = 50000

os.makedirs(MODEL_DIR, exist_ok=True)

# Constrained proxy model — forces the model to rely on feature engineering
PROXY_PARAMS = {
    'n_estimators': 200,
    'learning_rate': 0.1,
    'num_leaves': 31,
    'max_depth': 5,            # CRITICAL: keep shallow!
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': -1,
    'n_jobs': -1
}


@contextlib.contextmanager
def suppress_output():
    """Suppress stdout/stderr from noisy preprocessing functions."""
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        yield


print("\u2705 Imports and configuration loaded.")
print(f"   Using preprocessing_pipeline: fit_preprocessing_pipeline / apply_fitted_pipeline")

✅ Gemini client initialized.
✅ Imports and configuration loaded.
   Using preprocessing_pipeline: fit_preprocessing_pipeline / apply_fitted_pipeline


## Step 1: Leakage-Free Evaluation Engine

Core principle: **every transformation is fit on the training fold and applied to the validation fold**.

**Pipeline-based methods** call `fit_preprocessing_pipeline(X_tr, pipeline_steps)` which internally
uses `_fit_imputer_step`, `_fit_scaler_step`, etc. to learn parameters from X_tr only. Then
`apply_fitted_pipeline(X_va, fitted_steps)` replays those saved parameters on X_va.

**Custom methods** implement the same fit-on-train / transform-on-val logic manually for
preprocessing techniques not supported by the pipeline.

In [2]:
# ============================================================
# Leakage-Free Evaluation Engine
# ============================================================

def evaluate_method(X, y, method_fn=None):
    """
    5-Fold Stratified CV with leakage-free preprocessing.
    All transformations are applied WITHIN each fold.

    Args:
        X: Feature DataFrame
        y: Target Series (integer-encoded)
        method_fn: callable(X_tr, X_va, y_tr) -> (X_tr, X_va), or None for baseline

    Returns:
        (mean_accuracy, std_accuracy) or (None, None) on failure
    """
    try:
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        splits = list(kf.split(X, y))
    except ValueError:
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        splits = list(kf.split(X))

    scores = []

    for train_idx, val_idx in splits:
        X_tr = X.iloc[train_idx].copy().reset_index(drop=True)
        X_va = X.iloc[val_idx].copy().reset_index(drop=True)
        y_tr = y.iloc[train_idx].copy().reset_index(drop=True)
        y_va = y.iloc[val_idx].copy().reset_index(drop=True)

        # --- Apply method-specific transformation (within fold) ---
        if method_fn is not None:
            try:
                X_tr, X_va = method_fn(X_tr, X_va, y_tr)
            except Exception:
                return None, None

        # --- Encode remaining categorical columns (train-only mapping) ---
        cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
        for c in cat_cols:
            uniques = X_tr[c].astype(str).unique()
            mapping = {v: i for i, v in enumerate(uniques)}
            X_tr[c] = X_tr[c].astype(str).map(mapping).fillna(-1).astype(float)
            X_va[c] = X_va[c].astype(str).map(mapping).fillna(-1).astype(float)

        # --- Ensure all columns are numeric ---
        for c in X_tr.columns:
            if not np.issubdtype(X_tr[c].dtype, np.number):
                X_tr[c] = pd.to_numeric(X_tr[c], errors='coerce')
                X_va[c] = pd.to_numeric(X_va[c], errors='coerce')

        # --- Train with early stopping (constrained proxy model) ---
        try:
            model = lgb.LGBMClassifier(**PROXY_PARAMS)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[early_stopping(stopping_rounds=30, verbose=False)]
            )
            preds = model.predict(X_va, num_iteration=model.best_iteration_)
            scores.append(accuracy_score(y_va, preds))
        except Exception:
            return None, None

    return np.mean(scores), np.std(scores)


# ============================================================
# PIPELINE-BASED METHODS
# These use fit_preprocessing_pipeline() on X_tr and
# apply_fitted_pipeline() on X_va, leveraging the project's
# existing preprocessing_pipeline.py infrastructure.
# ============================================================

def method_impute_median(X_tr, X_va, y_tr):
    """Impute missing values via preprocessing_pipeline.
    
    Pipeline calls:
      - impute_median (numeric) -> _fit_imputer_step / _transform_imputer_step
      - impute_mode (categorical) -> _fit_imputer_step / _transform_imputer_step
    """
    num_cols = [c for c in X_tr.select_dtypes(include=[np.number]).columns
                if X_tr[c].isna().any() or X_va[c].isna().any()]
    cat_cols = [c for c in X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
                if X_tr[c].isna().any() or X_va[c].isna().any()]

    if not num_cols and not cat_cols:
        return X_tr, X_va

    pipeline = []
    if num_cols:
        pipeline.append({"function_to_call": "impute_median", "kwargs": {"columns": num_cols}})
    if cat_cols:
        pipeline.append({"function_to_call": "impute_mode", "kwargs": {"columns": cat_cols}})

    with suppress_output():
        X_tr, fitted = fit_preprocessing_pipeline(X_tr, pipeline)
        X_va = apply_fitted_pipeline(X_va, fitted)
    return X_tr, X_va


def method_missing_indicator(X_tr, X_va, y_tr):
    """Add missing indicators via preprocessing_pipeline.
    
    Pipeline calls:
      - add_missing_indicator -> _fit_indicator_step / _transform_indicator_step
    """
    cols_with_na = [c for c in X_tr.columns
                    if X_tr[c].isna().any() or X_va[c].isna().any()]
    if not cols_with_na:
        return X_tr, X_va

    pipeline = [{"function_to_call": "add_missing_indicator", "kwargs": {"columns": cols_with_na}}]
    with suppress_output():
        X_tr, fitted = fit_preprocessing_pipeline(X_tr, pipeline)
        X_va = apply_fitted_pipeline(X_va, fitted)
    return X_tr, X_va


def method_standard_scaler(X_tr, X_va, y_tr):
    """Standardize numerics via preprocessing_pipeline.
    
    Pipeline calls:
      - impute_median -> _fit_imputer_step (fill NaN with train median)
      - standard_scaler -> _fit_scaler_step / _transform_scaler_step
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        return X_tr, X_va

    pipeline = [
        {"function_to_call": "impute_median", "kwargs": {"columns": num_cols}},
        {"function_to_call": "standard_scaler", "kwargs": {"column": num_cols}},
    ]
    with suppress_output():
        X_tr, fitted = fit_preprocessing_pipeline(X_tr, pipeline)
        X_va = apply_fitted_pipeline(X_va, fitted)
    return X_tr, X_va


def method_robust_scaler(X_tr, X_va, y_tr):
    """Robust scaling via preprocessing_pipeline.
    
    Pipeline calls:
      - impute_median -> _fit_imputer_step
      - robust_scaler -> _fit_robust_scaler_step / _transform_robust_scaler_step
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        return X_tr, X_va

    pipeline = [
        {"function_to_call": "impute_median", "kwargs": {"columns": num_cols}},
        {"function_to_call": "robust_scaler", "kwargs": {"column": num_cols}},
    ]
    with suppress_output():
        X_tr, fitted = fit_preprocessing_pipeline(X_tr, pipeline)
        X_va = apply_fitted_pipeline(X_va, fitted)
    return X_tr, X_va


def method_yeo_johnson(X_tr, X_va, y_tr):
    """Yeo-Johnson power transform via preprocessing_pipeline.
    
    Pipeline calls:
      - impute_median -> _fit_imputer_step
      - apply_power_transform (per column) -> _fit_power_transform_step / _transform_power_transform_step
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        return X_tr, X_va

    pipeline = [{"function_to_call": "impute_median", "kwargs": {"columns": num_cols}}]
    for c in num_cols:
        pipeline.append({
            "function_to_call": "apply_power_transform",
            "kwargs": {"column": c, "method": "yeo-johnson"}
        })

    with suppress_output():
        X_tr, fitted = fit_preprocessing_pipeline(X_tr, pipeline)
        X_va = apply_fitted_pipeline(X_va, fitted)
    return X_tr, X_va


def method_clip_outliers(X_tr, X_va, y_tr):
    """Clip outliers via preprocessing_pipeline.
    
    Pipeline calls:
      - clip_outliers_iqr -> _fit_clip_outliers_step / _transform_clip_outliers_step
      IQR bounds are computed from train fold only, then applied to val fold.
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        return X_tr, X_va

    pipeline = [{"function_to_call": "clip_outliers_iqr", "kwargs": {"column": num_cols}}]
    with suppress_output():
        X_tr, fitted = fit_preprocessing_pipeline(X_tr, pipeline)
        X_va = apply_fitted_pipeline(X_va, fitted)
    return X_tr, X_va


# ============================================================
# CUSTOM METHODS
# For preprocessing techniques where the pipeline does NOT
# provide fit/transform separation. These are manually
# leakage-free wrappers.
# ============================================================

def method_frequency_encoding(X_tr, X_va, y_tr):
    """Frequency encoding: frequencies computed from train only.
    
    Note: preprocessing_function.frequency_encode() operates globally on a DataFrame,
    and the pipeline treats it as a stateless step (re-runs on test data).
    This would cause leakage, so we split the logic manually.
    """
    cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
    for c in cat_cols:
        freq = X_tr[c].astype(str).value_counts(normalize=True)
        X_tr[c] = X_tr[c].astype(str).map(freq).fillna(0).astype(float)
        X_va[c] = X_va[c].astype(str).map(freq).fillna(0).astype(float)
    return X_tr, X_va


def method_target_encoding(X_tr, X_va, y_tr):
    """Smoothed target encoding (Bayesian) for categoricals.
    
    Note: Target encoding is not in the existing codebase.
    This is a custom implementation with Bayesian smoothing (m=10)
    to prevent overfitting on rare categories.
    """
    cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
    global_mean = y_tr.mean()
    m = 10  # smoothing factor
    for c in cat_cols:
        col_str = X_tr[c].astype(str)
        agg = y_tr.groupby(col_str).agg(['count', 'mean'])
        smooth = (agg['count'] * agg['mean'] + m * global_mean) / (agg['count'] + m)
        X_tr[c] = X_tr[c].astype(str).map(smooth).fillna(global_mean).astype(float)
        X_va[c] = X_va[c].astype(str).map(smooth).fillna(global_mean).astype(float)
    return X_tr, X_va


def method_onehot_encoding(X_tr, X_va, y_tr):
    """One-hot encode low-cardinality categoricals (<=10 unique).
    
    Note: preprocessing_function.one_hot_encode() uses pd.get_dummies which
    has no fit/transform separation (may create different columns on test).
    We use sklearn's OneHotEncoder with handle_unknown='ignore' instead.
    """
    cat_cols = X_tr.select_dtypes(include=['object', 'category', 'bool']).columns
    low_card = [c for c in cat_cols if X_tr[c].nunique() <= 10]
    if not low_card:
        return X_tr, X_va

    ohe = SklearnOHE(sparse_output=False, handle_unknown='ignore', drop='if_binary')
    tr_ohe = ohe.fit_transform(X_tr[low_card].astype(str))
    va_ohe = ohe.transform(X_va[low_card].astype(str))
    ohe_names = ohe.get_feature_names_out(low_card)

    X_tr = X_tr.drop(columns=low_card)
    X_va = X_va.drop(columns=low_card)
    X_tr = pd.concat([X_tr, pd.DataFrame(tr_ohe, columns=ohe_names)], axis=1)
    X_va = pd.concat([X_va, pd.DataFrame(va_ohe, columns=ohe_names)], axis=1)
    return X_tr, X_va


def method_log_transform(X_tr, X_va, y_tr):
    """Log-transform highly skewed (|skew|>1) numerical columns.
    
    Note: No corresponding pipeline function exists.
    Skewness threshold and offset are determined from train data only.
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns
    skewed = [c for c in num_cols
              if len(X_tr[c].dropna()) > 0 and abs(skew(X_tr[c].dropna())) > 1]
    if not skewed:
        return X_tr, X_va
    for c in skewed:
        offset = abs(X_tr[c].min()) + 1 if X_tr[c].min() <= 0 else 0
        X_tr[c] = np.log1p(X_tr[c] + offset)
        X_va[c] = np.log1p(X_va[c] + offset)
    return X_tr, X_va


def _llm_suggest_interactions(columns, stats_summary):
    """
    Ask Gemini to suggest meaningful feature interactions.
    
    Args:
        columns: list of numerical column names
        stats_summary: string of df.describe() output (from TRAIN only)
    
    Returns:
        list of dicts: [{"col_a": str, "col_b": str, "operation": str}, ...]
    """
    global gemini_client, _llm_suggestion_cache
    
    if gemini_client is None:
        return []
    
    # Cache key: frozenset of column names
    cache_key = frozenset(columns)
    if cache_key in _llm_suggestion_cache:
        return _llm_suggestion_cache[cache_key]
    
    prompt = f"""You are a feature engineering expert for machine learning.
Given a dataset with these numerical columns and their statistics:

Columns: {columns}

Statistics (from training data only):
{stats_summary}

Suggest up to 15 meaningful pairwise feature interactions.
For each, specify which two columns to combine and which operation to use.

Rules:
- Only use columns from the list above
- Operations: "product" (A*B), "ratio" (A/B), "difference" (A-B), "sum" (A+B)
- Choose pairs and operations that could capture real-world relationships
- Prefer ratio for columns with different scales (e.g., amount/count = per-unit)
- Prefer product for columns that together form a meaningful concept
- Prefer difference for columns measuring similar things

Return ONLY a JSON array, no other text. Example format:
[
  {{"col_a": "column1", "col_b": "column2", "operation": "ratio"}},
  {{"col_a": "column3", "col_b": "column4", "operation": "product"}}
]"""

    try:
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt,
            config={
                "temperature": 0,
                "max_output_tokens": 2048,
            }
        )
        
        # Parse JSON from response
        text = response.text.strip()
        # Remove markdown code block if present
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
        
        suggestions = json.loads(text)
        
        # Validate: only keep suggestions with valid columns and operations
        valid_ops = {'product', 'ratio', 'difference', 'sum'}
        col_set = set(columns)
        validated = []
        for s in suggestions:
            if (isinstance(s, dict) and
                s.get('col_a') in col_set and
                s.get('col_b') in col_set and
                s.get('operation') in valid_ops and
                s['col_a'] != s['col_b']):
                validated.append(s)
        
        _llm_suggestion_cache[cache_key] = validated
        return validated
        
    except Exception as e:
        _llm_suggestion_cache[cache_key] = []
        return []


def _generate_llm_features(X, suggestions):
    """Generate features from LLM suggestions (applies to both train and val)."""
    candidates = {}
    for s in suggestions:
        a, b, op = s['col_a'], s['col_b'], s['operation']
        if a not in X.columns or b not in X.columns:
            continue
        if op == 'product':
            candidates[f'{a}_x_{b}'] = X[a] * X[b]
        elif op == 'ratio':
            safe_b = X[b].replace(0, np.nan)
            candidates[f'{a}_div_{b}'] = X[a] / safe_b
        elif op == 'difference':
            candidates[f'{a}_minus_{b}'] = X[a] - X[b]
        elif op == 'sum':
            candidates[f'{a}_plus_{b}'] = X[a] + X[b]
    return pd.DataFrame(candidates, index=X.index) if candidates else pd.DataFrame()


def method_arithmetic_interactions(X_tr, X_va, y_tr):
    """LLM-enhanced interaction features (v4).
    
    Hybrid approach — merges two sources of feature interaction ideas:
    1. **Gemini LLM**: Semantic suggestions based on column names + statistics
       (called once per dataset, cached across folds)
    2. **Correlation analysis**: Statistical high-correlation pairs from train data
       (using preprocessing_function.py internals)
    
    All operations are leakage-free: statistics from TRAIN only.
    Falls back to correlation-only if LLM is unavailable.
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 2:
        return X_tr, X_va

    # Fill NaN with train median (leakage-free)
    for c in num_cols:
        med = X_tr[c].median()
        X_tr[c] = X_tr[c].fillna(med)
        X_va[c] = X_va[c].fillna(med)

    all_tr_features = []
    all_va_features = []

    try:
        # ============================================================
        # Source 1: LLM Suggestions (semantic, column-name-aware)
        # ============================================================
        stats_str = X_tr[num_cols].describe().to_string()
        llm_suggestions = _llm_suggest_interactions(num_cols, stats_str)

        if llm_suggestions:
            tr_llm = _generate_llm_features(X_tr, llm_suggestions)
            va_llm = _generate_llm_features(X_va, llm_suggestions)

            if not tr_llm.empty:
                all_tr_features.append(tr_llm)
                all_va_features.append(va_llm)

        # ============================================================
        # Source 2: Correlation Analysis (statistical, data-driven)
        # ============================================================
        corr_matrix = X_tr[num_cols].corr()

        high_corr_pairs = _find_high_corr_pairs(
            corr_matrix, num_cols, target_column=None, threshold=0.3
        )

        if not high_corr_pairs:
            high_corr_pairs = _find_high_corr_pairs(
                corr_matrix, num_cols, target_column=None, threshold=0.0
            )
            high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
            high_corr_pairs = high_corr_pairs[:20]

        if high_corr_pairs:
            if len(high_corr_pairs) > 200:
                high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
                high_corr_pairs = high_corr_pairs[:200]

            feature_types = ['product', 'ratio']
            tr_corr = _generate_candidate_features(X_tr, high_corr_pairs, feature_types)
            va_corr = _generate_candidate_features(X_va, high_corr_pairs, feature_types)

            if not tr_corr.empty:
                all_tr_features.append(tr_corr)
                all_va_features.append(va_corr)

        # ============================================================
        # Merge, deduplicate, and filter
        # ============================================================
        if not all_tr_features:
            return X_tr, X_va

        tr_combined = pd.concat(all_tr_features, axis=1)
        va_combined = pd.concat(all_va_features, axis=1)

        # Remove duplicate column names (LLM and corr may suggest same pair)
        tr_combined = tr_combined.loc[:, ~tr_combined.columns.duplicated()]
        va_combined = va_combined.loc[:, ~va_combined.columns.duplicated()]

        # Quality filter based on TRAIN statistics
        with suppress_output():
            tr_filtered = _apply_basic_filter(tr_combined, min_cardinality=3, min_variance=1e-6)

        if tr_filtered.empty:
            return X_tr, X_va

        kept_cols = tr_filtered.columns.tolist()

        # Cap at 60 features (LLM may add extras beyond corr-only)
        if len(kept_cols) > 60:
            var_order = tr_filtered.var().sort_values(ascending=False)
            kept_cols = var_order.head(60).index.tolist()

        va_filtered = va_combined.reindex(columns=kept_cols, fill_value=0)

        X_tr = pd.concat([X_tr, tr_filtered[kept_cols].reset_index(drop=True)], axis=1)
        X_va = pd.concat([X_va, va_filtered.reset_index(drop=True)], axis=1)

    except Exception:
        # Ultimate fallback: simple top-5 variance product
        top = X_tr[num_cols].var().sort_values(ascending=False).head(5).index
        for i, a in enumerate(top):
            for b in top[i+1:]:
                name = f'{a}_x_{b}'
                X_tr[name] = X_tr[a] * X_tr[b]
                X_va[name] = X_va[a] * X_va[b]

    return X_tr, X_va


def method_fastica(X_tr, X_va, y_tr):
    """FastICA dimensionality reduction using preprocessing_function.py internals.
    
    Leakage-free implementation:
    1. Fill NaN with TRAIN median
    2. Fit StandardScaler on TRAIN only
    3. Fit FastICA on TRAIN only
    4. Transform both train and val with fitted objects
    5. Use hybrid mode: keep high-importance original features, replace low-importance
    6. Add ICA interaction features using _create_ica_interactions
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 3:
        return X_tr, X_va

    # --- Step 1: Fill NaN with train median ---
    for c in num_cols:
        med = X_tr[c].median()
        X_tr[c] = X_tr[c].fillna(med)
        X_va[c] = X_va[c].fillna(med)

    # --- Step 2: Auto-select n_components ---
    n_components = min(len(num_cols), len(X_tr) // 2, 10)
    n_components = max(2, n_components)
    if n_components >= len(num_cols):
        n_components = len(num_cols) - 1

    try:
        # --- Step 3: Fit StandardScaler on TRAIN only ---
        scaler = SklearnStdScaler()
        X_tr_scaled = scaler.fit_transform(X_tr[num_cols])
        X_va_scaled = scaler.transform(X_va[num_cols])

        # --- Step 4: Fit FastICA on TRAIN only ---
        ica = FastICA(
            n_components=n_components,
            random_state=42,
            max_iter=1000,
            whiten='unit-variance'
        )
        tr_ica = ica.fit_transform(X_tr_scaled)
        va_ica = ica.transform(X_va_scaled)

        component_names = [f'ICA_{i}' for i in range(n_components)]
        tr_ica_df = pd.DataFrame(tr_ica, columns=component_names, index=X_tr.index)
        va_ica_df = pd.DataFrame(va_ica, columns=component_names, index=X_va.index)

        # --- Step 5: Hybrid mode — replace low-variance features, keep high-variance ---
        replace_ratio = 0.4
        n_to_replace = max(1, int(len(num_cols) * replace_ratio))
        n_to_replace = min(n_to_replace, len(num_cols) - 1)

        # Sort by variance (from train): lowest variance = least important = replace first
        var_order = X_tr[num_cols].var().sort_values()
        cols_to_replace = var_order.head(n_to_replace).index.tolist()
        cols_to_keep = var_order.tail(len(num_cols) - n_to_replace).index.tolist()

        non_num_cols = [c for c in X_tr.columns if c not in num_cols]

        # Build result: non-numeric + kept numeric + ICA components
        tr_parts = []
        va_parts = []
        if non_num_cols:
            tr_parts.append(X_tr[non_num_cols].reset_index(drop=True))
            va_parts.append(X_va[non_num_cols].reset_index(drop=True))
        if cols_to_keep:
            tr_parts.append(X_tr[cols_to_keep].reset_index(drop=True))
            va_parts.append(X_va[cols_to_keep].reset_index(drop=True))
        tr_parts.append(tr_ica_df.reset_index(drop=True))
        va_parts.append(va_ica_df.reset_index(drop=True))

        X_tr = pd.concat(tr_parts, axis=1)
        X_va = pd.concat(va_parts, axis=1)

        # --- Step 6: Add ICA interaction features ---
        tr_interactions = _create_ica_interactions(tr_ica_df, n_interactions=min(5, n_components))
        va_interactions = _create_ica_interactions(va_ica_df, n_interactions=min(5, n_components))
        if not tr_interactions.empty:
            X_tr = pd.concat([X_tr.reset_index(drop=True), tr_interactions.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), va_interactions.reset_index(drop=True)], axis=1)

    except Exception:
        # If FastICA fails (e.g. convergence), return unchanged
        pass

    return X_tr, X_va


def method_pca_95(X_tr, X_va, y_tr):
    """PCA retaining 95% variance.
    
    Hybrid: uses preprocessing_pipeline for imputation + scaling,
    then sklearn PCA for dimensionality reduction (not in pipeline).
    """
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 3:
        return X_tr, X_va

    # Use pipeline for imputation + scaling
    pipeline = [
        {"function_to_call": "impute_median", "kwargs": {"columns": num_cols}},
        {"function_to_call": "standard_scaler", "kwargs": {"column": num_cols}},
    ]
    with suppress_output():
        X_tr, fitted = fit_preprocessing_pipeline(X_tr, pipeline)
        X_va = apply_fitted_pipeline(X_va, fitted)

    # PCA (sklearn — not available in pipeline)
    pca = PCA(n_components=0.95, random_state=42)
    tr_pca = pca.fit_transform(X_tr[num_cols])
    va_pca = pca.transform(X_va[num_cols])
    X_tr = X_tr.drop(columns=num_cols)
    X_va = X_va.drop(columns=num_cols)
    pca_names = [f'PC_{i}' for i in range(tr_pca.shape[1])]
    X_tr = pd.concat([X_tr, pd.DataFrame(tr_pca, columns=pca_names)], axis=1)
    X_va = pd.concat([X_va, pd.DataFrame(va_pca, columns=pca_names)], axis=1)
    return X_tr, X_va


# ============================================================
# Method Registry & Applicability Check
# ============================================================

ALL_METHODS = {
    # Pipeline-based methods (use fit_preprocessing_pipeline / apply_fitted_pipeline)
    'ImputeMedian':             method_impute_median,
    'MissingIndicator':         method_missing_indicator,
    'StandardScaler':           method_standard_scaler,
    'RobustScaler':             method_robust_scaler,
    'YeoJohnson':               method_yeo_johnson,
    'ClipOutliers':             method_clip_outliers,
    # Custom methods (manually leakage-free wrappers)
    'FrequencyEncoding':        method_frequency_encoding,
    'TargetEncoding':           method_target_encoding,
    'OneHotEncoding':           method_onehot_encoding,
    'LogTransform':             method_log_transform,
    # Feature engineering (using preprocessing_function.py internals)
    'ArithmeticInteractions':   method_arithmetic_interactions,
    'FastICA':                  method_fastica,
    'PCA_95':                   method_pca_95,
}


def get_applicable_methods(X):
    """Determine which methods are applicable to this dataset."""
    cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns
    num_cols = X.select_dtypes(include=[np.number]).columns
    has_missing = X.isna().any().any()
    has_cat = len(cat_cols) > 0
    has_num = len(num_cols) > 0
    has_low_card_cat = any(X[c].nunique() <= 10 for c in cat_cols) if has_cat else False
    has_skewed = False
    if has_num:
        for c in num_cols:
            clean = X[c].dropna()
            if len(clean) > 2 and abs(skew(clean)) > 1:
                has_skewed = True
                break

    applicable = {}

    # Pipeline-based methods
    if has_missing:
        applicable['ImputeMedian'] = method_impute_median
        applicable['MissingIndicator'] = method_missing_indicator
    if has_num:
        applicable['StandardScaler'] = method_standard_scaler
        applicable['RobustScaler'] = method_robust_scaler
        applicable['YeoJohnson'] = method_yeo_johnson
        applicable['ClipOutliers'] = method_clip_outliers

    # Custom methods
    if has_cat:
        applicable['FrequencyEncoding'] = method_frequency_encoding
        applicable['TargetEncoding'] = method_target_encoding
    if has_low_card_cat:
        applicable['OneHotEncoding'] = method_onehot_encoding
    if has_skewed:
        applicable['LogTransform'] = method_log_transform
    if has_num and len(num_cols) >= 2:
        applicable['ArithmeticInteractions'] = method_arithmetic_interactions
    if has_num and len(num_cols) >= 3:
        applicable['FastICA'] = method_fastica
        applicable['PCA_95'] = method_pca_95

    return applicable


# Summary
pipeline_methods = ['ImputeMedian', 'MissingIndicator', 'StandardScaler',
                    'RobustScaler', 'YeoJohnson', 'ClipOutliers']
func_methods = ['ArithmeticInteractions', 'FastICA']
custom_methods = ['FrequencyEncoding', 'TargetEncoding', 'OneHotEncoding',
                  'LogTransform', 'ArithmeticInteractions', 'PCA_95']

print(f"\u2705 Evaluation engine loaded. {len(ALL_METHODS)} methods available.")
print(f"   Pipeline-based ({len(pipeline_methods)}): {pipeline_methods}")
print(f"   Custom wrappers ({len(custom_methods)}): {custom_methods}")

✅ Evaluation engine loaded. 13 methods available.
   Pipeline-based (6): ['ImputeMedian', 'MissingIndicator', 'StandardScaler', 'RobustScaler', 'YeoJohnson', 'ClipOutliers']
   Custom wrappers (6): ['FrequencyEncoding', 'TargetEncoding', 'OneHotEncoding', 'LogTransform', 'ArithmeticInteractions', 'PCA_95']


## Step 2: Meta-Feature Extraction (70 features)

Dataset-level meta-features that describe the characteristics of a dataset:
- **Generic (7)**: n_instances, n_attributes, dimensionality, missing values
- **Class (6)**: n_classes, class entropy, minority/majority class
- **Continuous (30)**: statistics of means, stds, skewness, kurtosis across numerical columns
- **Categorical (27)**: statistics of entropy, mutual info, distinct values across categorical columns

In [3]:
# ============================================================
# Meta-Feature Extraction (70 features)
# ============================================================

def _get_quartiles(series):
    if len(series) == 0:
        return 0, 0, 0
    return series.quantile([0.25, 0.5, 0.75]).tolist()


def _calc_stats(series, prefix):
    """Min, Mean, Max, Std, Q1, Q2, Q3 for a series."""
    if len(series) == 0:
        return {f'Min_{prefix}': 0, f'Mean_{prefix}': 0, f'Max_{prefix}': 0,
                f'Std_{prefix}': 0, f'Q1_{prefix}': 0, f'Q2_{prefix}': 0, f'Q3_{prefix}': 0}
    q1, q2, q3 = _get_quartiles(series)
    return {
        f'Min_{prefix}': series.min(),
        f'Mean_{prefix}': series.mean(),
        f'Max_{prefix}': series.max(),
        f'Std_{prefix}': series.std(ddof=1) if len(series) > 1 else 0,
        f'Q1_{prefix}': q1, f'Q2_{prefix}': q2, f'Q3_{prefix}': q3
    }


def extract_meta_features(X, y, problem_type='classification'):
    """
    Extract 70 meta-features from a dataset.

    Args:
        X: Feature DataFrame (without target)
        y: Target Series
        problem_type: 'classification' or 'regression'

    Returns:
        dict of meta-features
    """
    f = {}
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    n_inst = len(X)
    n_attr = len(X.columns)

    # ---- Generic Features (7) ----
    f['Number_of_Instances'] = n_inst
    f['Number_of_Attributes'] = n_attr
    f['Dimensionality'] = n_attr / n_inst if n_inst > 0 else 0

    missing_total = X.isnull().sum().sum()
    f['Number_of_Missing_Values'] = missing_total
    f['Percentage_of_Missing_Values'] = missing_total / (n_inst * n_attr) if n_inst * n_attr > 0 else 0

    rows_missing = X.isnull().any(axis=1).sum()
    f['Number_of_Instances_with_Missing_Values'] = rows_missing
    f['Percentage_of_Instances_with_Missing_Values'] = rows_missing / n_inst if n_inst > 0 else 0

    # ---- Class Features (6) ----
    if y is not None and len(y) > 0 and problem_type != 'regression':
        cc = y.value_counts()
        f['Number_of_Classes'] = len(cc)
        probs = cc / n_inst
        f['Class_Entropy'] = entropy(probs, base=2)
        f['Minority_Class_Size'] = cc.min()
        f['Majority_Class_Size'] = cc.max()
        f['Minority_Class_Percentage'] = cc.min() / n_inst
        f['Majority_Class_Percentage'] = cc.max() / n_inst
    else:
        for k in ['Number_of_Classes', 'Class_Entropy', 'Minority_Class_Size',
                   'Majority_Class_Size', 'Minority_Class_Percentage', 'Majority_Class_Percentage']:
            f[k] = 0

    # ---- Continuous Attribute Statistics (30) ----
    f['Number_of_Continuous_Attributes'] = len(num_cols)
    f['Percentage_of_Continuous_Attributes'] = len(num_cols) / n_attr if n_attr > 0 else 0

    if len(num_cols) > 0:
        X_num = X[num_cols]
        means  = X_num.mean()
        stds   = X_num.std()
        skews  = X_num.apply(lambda s: skew(s.dropna()) if len(s.dropna()) > 2 else 0)
        kurts  = X_num.apply(lambda s: kurtosis(s.dropna()) if len(s.dropna()) > 2 else 0)
        f.update(_calc_stats(means,  'Means_of_Continuous_Attributes'))
        f.update(_calc_stats(stds,   'Std_of_Continuous_Attributes'))
        f.update(_calc_stats(skews,  'Skewness_of_Continuous_Attributes'))
        f.update(_calc_stats(kurts,  'Kurtosis_of_Continuous_Attributes'))
    else:
        for stat in ['Means', 'Std', 'Skewness', 'Kurtosis']:
            f.update(_calc_stats(pd.Series([], dtype=float), f'{stat}_of_Continuous_Attributes'))

    # ---- Categorical & Binary Attribute Statistics (27) ----
    binary_cols = [c for c in X.columns if X[c].nunique() == 2]
    f['Number_of_Categorical_Attributes'] = len(cat_cols)
    f['Number_of_Binary_Attributes'] = len(binary_cols)
    f['Percentage_of_Categorical_Attributes'] = len(cat_cols) / n_attr if n_attr > 0 else 0
    f['Percentage_of_Binary_Attributes'] = len(binary_cols) / n_attr if n_attr > 0 else 0

    if len(cat_cols) > 0:
        entropies, mutual_infos, distinct_vals = [], [], []
        for c in cat_cols:
            vc = X[c].astype(str).value_counts(normalize=True)
            entropies.append(entropy(vc, base=2))
            distinct_vals.append(X[c].nunique())
            mutual_infos.append(0)  # simplified for speed
        f.update(_calc_stats(pd.Series(entropies),     'Attribute_Entropy'))
        f.update(_calc_stats(pd.Series(mutual_infos),  'Mutual_Information'))
        f.update(_calc_stats(pd.Series(distinct_vals, dtype=float), 'Attribute_Distinct_Values'))
        f['Equivalent_Number_of_Attributes'] = 0
        f['Noise_to_Signal_Ratio'] = 0
    else:
        for stat in ['Attribute_Entropy', 'Mutual_Information', 'Attribute_Distinct_Values']:
            f.update(_calc_stats(pd.Series([], dtype=float), stat))
        f['Equivalent_Number_of_Attributes'] = 0
        f['Noise_to_Signal_Ratio'] = 0

    return f


# Quick test
_test_X = pd.DataFrame({'a': [1,2,3,4,5], 'b': ['x','y','x','y','z']})
_test_y = pd.Series([0, 1, 0, 1, 0])
_feats = extract_meta_features(_test_X, _test_y)
print(f"\u2705 Meta-feature extraction loaded. Produces {len(_feats)} features.")

✅ Meta-feature extraction loaded. Produces 70 features.


## Step 3: Collect Performance Labels from CC18

For each dataset in OpenML-CC18:
1. Download X, y from OpenML
2. Extract 70 meta-features
3. Evaluate Baseline (no preprocessing) using leakage-free 5-Fold CV
4. Evaluate each applicable preprocessing method
5. Record score, delta, and method for meta-model training

**Resume-capable**: skips datasets already saved in CSV.

In [4]:
# ============================================================
# Data Collection from CC18
# ============================================================

print(f"\U0001f50d Loading OpenML-CC18 (Suite {SUITE_ID})...")
suite = openml.study.get_suite(SUITE_ID)
task_ids = list(suite.tasks)
print(f"   Found {len(task_ids)} tasks.")

# Load existing results for resume
if os.path.exists(RESULTS_FILE):
    existing_labels = pd.read_csv(RESULTS_FILE)
    processed_ids = set(existing_labels['dataset_id'].unique())
    all_labels = existing_labels.to_dict('records')
    print(f"   Resuming: {len(processed_ids)} datasets already done.")
else:
    all_labels = []
    processed_ids = set()

if os.path.exists(META_FEATURES_FILE):
    existing_meta = pd.read_csv(META_FEATURES_FILE)
    all_meta = existing_meta.to_dict('records')
else:
    all_meta = []

total = len(task_ids)
t_start = time.time()

for idx, task_id in enumerate(task_ids):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{total}] Task {task_id}")

    try:
        # --- Load dataset ---
        task = openml.tasks.get_task(task_id)
        dataset = task.get_dataset()
        dataset_id = dataset.id
        dataset_name = dataset.name
        target_name = task.target_name

        if dataset_id in processed_ids:
            print(f"   \u23ed\ufe0f  {dataset_name} \u2014 already processed")
            continue

        X, y_raw, _, _ = dataset.get_data(target=target_name)

        # --- Basic checks ---
        if len(X) > MAX_ROWS:
            print(f"   \u26a0\ufe0f  {dataset_name} \u2014 too large ({len(X)} rows), skipping")
            continue
        if len(X) < 50:
            print(f"   \u26a0\ufe0f  {dataset_name} \u2014 too small ({len(X)} rows), skipping")
            continue

        # --- Encode target ---
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y_raw.astype(str)), name='target')

        # Drop datetime columns (if any)
        dt_cols = X.select_dtypes(include=['datetime64', 'datetime64[ns]']).columns
        if len(dt_cols) > 0:
            X = X.drop(columns=dt_cols)

        n_classes = y.nunique()
        print(f"   \U0001f4ca {dataset_name} | {X.shape[0]} rows x {X.shape[1]} cols | {n_classes} classes")

        # --- Extract meta-features ---
        mf = extract_meta_features(X, y)
        mf['dataset_id'] = dataset_id
        mf['dataset_name'] = dataset_name
        all_meta.append(mf)

        # --- Evaluate Baseline ---
        t0 = time.time()
        base_score, base_std = evaluate_method(X, y, method_fn=None)
        t_base = time.time() - t0

        if base_score is None:
            print(f"   \u274c Baseline evaluation failed, skipping dataset")
            continue

        all_labels.append({
            'dataset_id': dataset_id, 'dataset_name': dataset_name,
            'method': 'Baseline', 'score': base_score,
            'score_std': base_std, 'time_seconds': t_base
        })
        print(f"   \u2705 Baseline: {base_score:.4f} (\u00b1{base_std:.4f}) [{t_base:.1f}s]")

        # --- Evaluate each applicable method ---
        applicable = get_applicable_methods(X)
        print(f"   \U0001f527 Testing {len(applicable)} methods...")

        for method_name, method_fn in applicable.items():
            t0 = time.time()
            score, std = evaluate_method(X, y, method_fn)
            elapsed = time.time() - t0

            if score is not None:
                delta = score - base_score
                all_labels.append({
                    'dataset_id': dataset_id, 'dataset_name': dataset_name,
                    'method': method_name, 'score': score,
                    'score_std': std, 'time_seconds': elapsed
                })
                icon = '\u2705' if delta >= 0 else '\u274c'
                print(f"      {icon} {method_name:25s} | Acc={score:.4f} | \u0394={delta:+.4f} [{elapsed:.1f}s]")
            else:
                print(f"      \u26a0\ufe0f  {method_name:25s} | FAILED [{elapsed:.1f}s]")

        # --- Save intermediate results ---
        processed_ids.add(dataset_id)
        pd.DataFrame(all_labels).to_csv(RESULTS_FILE, index=False)
        pd.DataFrame(all_meta).to_csv(META_FEATURES_FILE, index=False)

        elapsed_total = time.time() - t_start
        print(f"   \U0001f4be Saved. Total time: {elapsed_total/60:.1f} min")

    except Exception as e:
        print(f"   \u274c Error: {e}")
        continue

# --- Final Summary ---
df_labels = pd.DataFrame(all_labels)
df_meta = pd.DataFrame(all_meta)
print(f"\n{'='*60}")
print(f"\u2705 Data collection complete!")
print(f"   Labels:        {len(df_labels)} rows ({df_labels['dataset_id'].nunique()} datasets)")
print(f"   Meta-features: {len(df_meta)} datasets x {len(df_meta.columns)} columns")
print(f"   Methods tested per dataset (avg): {len(df_labels) / max(1, df_labels['dataset_id'].nunique()):.1f}")

🔍 Loading OpenML-CC18 (Suite 99)...
   Found 72 tasks.

[1/72] Task 3
   📊 kr-vs-kp | 3196 rows x 36 cols | 2 classes
   ✅ Baseline: 0.9944 (±0.0025) [7.2s]
   🔧 Testing 3 methods...
      ❌ FrequencyEncoding         | Acc=0.9941 | Δ=-0.0003 [5.7s]
      ✅ TargetEncoding            | Acc=0.9950 | Δ=+0.0006 [5.8s]
      ❌ OneHotEncoding            | Acc=0.9937 | Δ=-0.0006 [5.6s]
   💾 Saved. Total time: 0.4 min

[2/72] Task 6
   📊 letter | 20000 rows x 16 cols | 26 classes
   ✅ Baseline: 0.9658 (±0.0007) [148.8s]
   🔧 Testing 8 methods...
      ✅ StandardScaler            | Acc=0.9660 | Δ=+0.0001 [122.2s]
      ✅ RobustScaler              | Acc=0.9660 | Δ=+0.0001 [116.7s]
      ✅ YeoJohnson                | Acc=0.9658 | Δ=+0.0000 [117.4s]
      ❌ ClipOutliers              | Acc=0.9644 | Δ=-0.0015 [122.6s]
      ✅ LogTransform              | Acc=0.9658 | Δ=+0.0000 [130.5s]
      ✅ ArithmeticInteractions    | Acc=0.9701 | Δ=+0.0043 [128.0s]
      ❌ FastICA                   | Acc=0.9551 | 

## Step 4: Train Meta-Model Regressors

For each preprocessing method:
1. Compute **gain = method_score - baseline_score** for each dataset
2. Train a LightGBM regressor: **X = 70 meta-features, y = gain**
3. Evaluate with 5-Fold CV using **Win Accuracy** (correctly predict gain > 0 or gain < 0)
4. Save model to disk

In [5]:
# ============================================================
# Meta-Model Training (Regression)
# ============================================================

# Load collected data
df_labels = pd.read_csv(RESULTS_FILE)
df_meta   = pd.read_csv(META_FEATURES_FILE)

print(f"\U0001f4c2 Loaded {len(df_labels)} labels, {len(df_meta)} datasets")

# Compute gain
baseline = df_labels[df_labels['method'] == 'Baseline'].set_index('dataset_id')['score']
df_labels['baseline_score'] = df_labels['dataset_id'].map(baseline)
df_labels['gain'] = df_labels['score'] - df_labels['baseline_score']
df_labels = df_labels.dropna(subset=['gain'])

# Merge with meta-features
full_df = pd.merge(df_labels, df_meta, on='dataset_id', how='inner')

# Identify feature columns (exclude metadata)
exclude = {'dataset_id', 'dataset_name', 'dataset_name_x', 'dataset_name_y',
           'method', 'score', 'score_std', 'time_seconds', 'baseline_score', 'gain'}
feature_cols = [c for c in full_df.select_dtypes(include=[np.number]).columns
                if c not in exclude]

print(f"\U0001f9e0 Meta-features for training: {len(feature_cols)}")
print(f"\U0001f4ca Total samples: {len(full_df)} ({full_df['dataset_id'].nunique()} datasets)")

# Train one regressor per method
methods = [m for m in full_df['method'].unique() if m != 'Baseline']
results = []

print(f"\n\U0001f3cb\ufe0f Training {len(methods)} regressors...")
print("-" * 70)

for method in methods:
    mdf = full_df[full_df['method'] == method]

    if len(mdf) < 5:
        print(f"   \u26a0\ufe0f  {method:25s} | Skipped (only {len(mdf)} samples)")
        continue

    X_m = mdf[feature_cols].copy()
    y_m = mdf['gain'].copy()

    # Fill any NaN in meta-features
    X_m = X_m.fillna(0)

    model = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1)

    n_splits = min(5, len(mdf))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_pred_cv = cross_val_predict(model, X_m, y_m, cv=kf)

    # Final fit on all data
    model.fit(X_m, y_m)
    joblib.dump(model, f"{MODEL_DIR}/regressor_{method}.pkl")

    rmse = np.sqrt(mean_squared_error(y_m, y_pred_cv))
    win_acc = np.mean((y_m > 0).values == (y_pred_cv > 0))

    results.append({
        'Method': method,
        'Samples': len(mdf),
        'RMSE': round(rmse, 6),
        'Win_Accuracy': round(win_acc, 4),
        'Avg_Gain': round(y_m.mean(), 6)
    })
    print(f"   \u2705 {method:25s} | N={len(mdf):3d} | WinAcc={win_acc:.2%} | AvgGain={y_m.mean():+.4f}")

# Save feature column names for inference
joblib.dump(feature_cols, f"{MODEL_DIR}/feature_cols.pkl")

# Summary
res_df = pd.DataFrame(results).sort_values('Win_Accuracy', ascending=False)
print(f"\n{'='*70}")
print("\U0001f4ca Meta-Model Training Summary (sorted by Win Accuracy):")
print(res_df.to_string(index=False))
res_df.to_csv("meta_model_summary_v4.csv", index=False)

📂 Loaded 609 labels, 66 datasets
🧠 Meta-features for training: 70
📊 Total samples: 609 (66 datasets)

🏋️ Training 13 regressors...
----------------------------------------------------------------------
   ✅ FrequencyEncoding         | N= 22 | WinAcc=63.64% | AvgGain=-0.0022
   ✅ TargetEncoding            | N= 22 | WinAcc=40.91% | AvgGain=-0.0002
   ✅ OneHotEncoding            | N= 22 | WinAcc=40.91% | AvgGain=-0.0007
   ✅ StandardScaler            | N= 59 | WinAcc=54.24% | AvgGain=+0.0005
   ✅ RobustScaler              | N= 59 | WinAcc=44.07% | AvgGain=+0.0000
   ✅ YeoJohnson                | N= 59 | WinAcc=57.63% | AvgGain=-0.0006
   ✅ ClipOutliers              | N= 59 | WinAcc=52.54% | AvgGain=-0.0117
   ✅ LogTransform              | N= 52 | WinAcc=30.77% | AvgGain=+0.0003
   ✅ ArithmeticInteractions    | N= 58 | WinAcc=48.28% | AvgGain=+0.0025
   ✅ FastICA                   | N= 57 | WinAcc=61.40% | AvgGain=-0.0032
   ✅ PCA_95                    | N= 56 | WinAcc=83.93% | AvgGain=-0.

## Step 5: Inference — Recommend Preprocessing for a New Dataset

Given a new dataset:
1. Extract 70 meta-features
2. Load all trained regressors
3. Predict gain for each preprocessing method
4. Recommend methods with positive predicted gain

In [6]:
# ============================================================
# Inference Pipeline
# ============================================================

def recommend_preprocessing(df, target_col=None, problem_type='classification', top_k=5):
    """
    Recommend preprocessing methods for a new dataset.

    Args:
        df: Input DataFrame (with or without target column)
        target_col: Name of the target column
        problem_type: 'classification' or 'regression'
        top_k: Number of top recommendations to display

    Returns:
        DataFrame with predicted gains for each method
    """
    print("\U0001f916 Meta-Model Inference (Regression)...")

    if not os.path.exists(MODEL_DIR):
        print(f"\u274c Model directory '{MODEL_DIR}' not found. Train models first (Step 4).")
        return None

    # 1. Separate X and y
    if target_col and target_col in df.columns:
        X = df.drop(columns=[target_col])
        y = df[target_col]
    else:
        X = df
        y = pd.Series(dtype=float)

    # 2. Extract meta-features
    print("   Extracting meta-features...")
    mf = extract_meta_features(X, y, problem_type)
    mf_df = pd.DataFrame([mf])
    print(f"   \u2705 Extracted {len(mf)} features")

    # 3. Load feature column names
    feature_cols = joblib.load(f"{MODEL_DIR}/feature_cols.pkl")

    # Align features (fill missing with 0)
    for c in feature_cols:
        if c not in mf_df.columns:
            mf_df[c] = 0
    X_meta = mf_df[feature_cols]

    # 4. Predict with each regressor
    predictions = []
    for fname in os.listdir(MODEL_DIR):
        if not fname.startswith('regressor_') or not fname.endswith('.pkl'):
            continue
        method = fname.replace('regressor_', '').replace('.pkl', '')
        try:
            model = joblib.load(os.path.join(MODEL_DIR, fname))
            if hasattr(model, 'feature_name_'):
                model_feats = model.feature_name_
                for c in model_feats:
                    if c not in mf_df.columns:
                        mf_df[c] = 0
                X_input = mf_df[model_feats]
            else:
                X_input = X_meta
            pred = model.predict(X_input)[0]
            predictions.append({
                'Method': method,
                'Predicted_Gain': pred,
                'Recommended': pred > 0
            })
        except Exception as e:
            print(f"   \u26a0\ufe0f Error with {method}: {e}")

    if not predictions:
        print("\u274c No predictions. Check if models exist.")
        return None

    result = pd.DataFrame(predictions).sort_values('Predicted_Gain', ascending=False)

    # 5. Display
    print(f"\n\U0001f4ca Top-{top_k} Preprocessing Recommendations:")
    print("-" * 55)
    for _, row in result.head(top_k).iterrows():
        icon = '\u2705' if row['Recommended'] else '\u274c'
        print(f"   {icon} {row['Method']:25s} | Gain: {row['Predicted_Gain']:+.4f}")
    print("-" * 55)

    rec = result[result['Recommended']]
    if len(rec) > 0:
        print(f"\n\U0001f3c6 Top Recommended: {rec.iloc[0]['Method']}")
    else:
        print("\n\u26a0\ufe0f No method predicted to improve performance. Use Baseline.")

    return result


print("\u2705 Inference pipeline loaded.")

✅ Inference pipeline loaded.


## Demo: Run Inference on a Test Dataset

In [7]:
# Create a synthetic test dataset for demonstration
np.random.seed(42)
df_demo = pd.DataFrame({
    'age':       np.random.randint(18, 80, 500),
    'income':    np.random.exponential(50000, 500),
    'score':     np.random.normal(100, 15, 500),
    'city':      np.random.choice(['Berlin', 'Munich', 'Hamburg', 'Frankfurt'], 500),
    'category':  np.random.choice(['A', 'B', 'C', 'D', 'E'], 500),
    'has_debt':  np.random.choice([0, 1], 500),
    'target':    np.random.randint(0, 2, 500)
})
# Add some missing values
df_demo.loc[df_demo.sample(50, random_state=1).index, 'income'] = np.nan
df_demo.loc[df_demo.sample(30, random_state=2).index, 'city'] = np.nan

print("\U0001f4cb Demo dataset:")
print(f"   Shape: {df_demo.shape}")
print(f"   Missing values: {df_demo.isnull().sum().sum()}")
print(f"   Columns: {list(df_demo.columns)}\n")

# Run inference
result = recommend_preprocessing(df_demo, target_col='target')

📋 Demo dataset:
   Shape: (500, 7)
   Missing values: 80
   Columns: ['age', 'income', 'score', 'city', 'category', 'has_debt', 'target']

🤖 Meta-Model Inference (Regression)...
   Extracting meta-features...
   ✅ Extracted 70 features

📊 Top-5 Preprocessing Recommendations:
-------------------------------------------------------
   ✅ ArithmeticInteractions    | Gain: +0.0069
   ❌ MissingIndicator          | Gain: -0.0001
   ❌ TargetEncoding            | Gain: -0.0002
   ❌ LogTransform              | Gain: -0.0003
   ❌ OneHotEncoding            | Gain: -0.0007
-------------------------------------------------------

🏆 Top Recommended: ArithmeticInteractions


## Step 6: Test with Real Datasets

Run inference on 8 real-world test datasets from the project root directory.
These datasets were NOT used during meta-model training (CC18 only), so they serve as
a genuine out-of-distribution test of the meta-model's generalization ability.

| Dataset | Rows | Cols | Classes | Notable |
|---------|------|------|---------|---------|
| maternal_health_risk | 1,014 | 7 | 3 | Small, clean, numeric-heavy |
| MIC | 1,699 | 112 | 8 | High dimensionality, massive missing values |
| splice | 3,190 | 61 | 3 | All categorical |
| hiva_agnostic | 3,845 | 1,618 | 3 | Ultra-high dimensionality |
| E-CommerceShippingData | 10,999 | 11 | 2 | Mixed types |
| online_shoppers_intention | 12,330 | 18 | 2 | Mixed types |
| Amazon_employee_access | 32,769 | 10 | 2 | All numeric (IDs), imbalanced |
| APSFailure | 76,000 | 171 | 2 | Large, massive missing values |

In [8]:
# ============================================================
# Test with Real Datasets
# ============================================================

# ============================================================
# Test with Real Datasets from testset/ folder
# ============================================================

TESTSET_DIR = "testset"

TEST_DATASETS = [
    {"file": "maternal_health_risk.csv",      "target": "RiskLevel"},
    {"file": "MIC.csv",                       "target": "LET_IS"},
    {"file": "splice.csv",                    "target": "SiteType"},
    {"file": "hiva_agnostic.csv",             "target": "CompoundActivity"},
    {"file": "E-CommereShippingData.csv",     "target": "ArrivedLate"},
    {"file": "online_shoppers_intention.csv", "target": "Revenue"},
    {"file": "Amazon_employee_access.csv",    "target": "ResourceApproved"},
    {"file": "APSFailure.csv",                "target": "AirPressureSystemFailure"},
]

all_test_results = []

print("=" * 70)
print("  REAL-WORLD DATASET TEST (Out-of-Distribution)")
print(f"  Model dir: {MODEL_DIR}")
print("=" * 70)

for info in TEST_DATASETS:
    filepath = os.path.join(TESTSET_DIR, info["file"])
    target_col = info["target"]
    dataset_name = info["file"].replace(".csv", "")

    if not os.path.exists(filepath):
        print(f"\n--- {info['file']} --- FILE NOT FOUND, skipping")
        continue

    print(f"\n{'=' * 60}")
    print(f"Dataset: {dataset_name}")
    print("-" * 60)

    try:
        df = pd.read_csv(filepath)
        n_rows, n_cols = df.shape
        n_missing = df.isna().sum().sum()
        n_classes = df[target_col].nunique() if target_col in df.columns else "?"

        print(f"  Shape:    {n_rows} rows x {n_cols} cols")
        print(f"  Target:   '{target_col}' ({n_classes} classes)")
        print(f"  Missing:  {n_missing} values ({n_missing / (n_rows * n_cols) * 100:.1f}%)")

        # Run inference
        result = recommend_preprocessing(df, target_col=target_col, top_k=5)

        if result is not None:
            result["dataset"] = dataset_name
            all_test_results.append(result)

    except Exception as e:
        print(f"  ERROR: {e}")

# ============================================================
# Summary Table
# ============================================================
if all_test_results:
    print(f"\n\n{'=' * 70}")
    print("  SUMMARY: Top Recommendation per Dataset")
    print("=" * 70)
    print(f"{'Dataset':<35s} {'Top Method':<25s} {'Gain':>10s}")
    print("-" * 70)

    for res in all_test_results:
        ds = res["dataset"].iloc[0]
        top = res.iloc[0]
        icon = "\u2705" if top["Recommended"] else "\u274c"
        print(f"{icon} {ds:<33s} {top['Method']:<25s} {top['Predicted_Gain']:+.4f}")

    print("-" * 70)
    print(f"\nTotal datasets tested: {len(all_test_results)}")
else:
    print("\nNo test results. Make sure models are trained (Step 4) and testset/ folder exists.")

  REAL-WORLD DATASET TEST (Out-of-Distribution)
  Model dir: meta_models_v4

Dataset: maternal_health_risk
------------------------------------------------------------
  Shape:    1014 rows x 7 cols
  Target:   'RiskLevel' (3 classes)
  Missing:  0 values (0.0%)
🤖 Meta-Model Inference (Regression)...
   Extracting meta-features...
   ✅ Extracted 70 features

📊 Top-5 Preprocessing Recommendations:
-------------------------------------------------------
   ✅ ClipOutliers              | Gain: +0.0207
   ✅ ArithmeticInteractions    | Gain: +0.0096
   ❌ MissingIndicator          | Gain: -0.0001
   ❌ TargetEncoding            | Gain: -0.0002
   ❌ LogTransform              | Gain: -0.0002
-------------------------------------------------------

🏆 Top Recommended: ClipOutliers

Dataset: MIC
------------------------------------------------------------
  Shape:    1699 rows x 112 cols
  Target:   'LET_IS' (8 classes)
  Missing:  15965 values (8.4%)
🤖 Meta-Model Inference (Regression)...
   Extra

## Step 7: Ground-Truth Validation

The meta-model **predicts** gain, but does the recommended preprocessing **actually** improve accuracy?

For each testset dataset:
1. Run **Baseline** 5-fold CV (no preprocessing)
2. Run each **individually recommended** method (positive predicted gain) with 5-fold CV
3. Build a **combined pipeline** from compatible recommended methods and run 5-fold CV
4. Compare predicted vs actual gain, and measure **direction accuracy** (did the meta-model correctly predict whether a method helps or hurts?)

**Combined pipeline logic** — mutually exclusive groups, pick highest predicted gain from each:
- Imputation → Indicator → Scaling/Transform → Encoding → Feature Engineering

In [9]:
# ============================================================
# Ground-Truth Validation on Testset
# ============================================================

MAX_VAL_ROWS = 10000  # Sample large datasets for reasonable runtime

# Mutually exclusive method groups (pick best from each)
SCALING_GROUP = {'StandardScaler', 'RobustScaler', 'YeoJohnson', 'LogTransform', 'ClipOutliers'}
ENCODING_GROUP = {'FrequencyEncoding', 'TargetEncoding', 'OneHotEncoding'}
FEATURE_GROUP = {'ArithmeticInteractions', 'FastICA', 'PCA_95'}
IMPUTE_GROUP = {'ImputeMedian'}
INDICATOR_GROUP = {'MissingIndicator'}

# Pipeline order: impute first, feature engineering last
PIPELINE_GROUPS = [
    ('imputation', IMPUTE_GROUP),
    ('indicator', INDICATOR_GROUP),
    ('scaling', SCALING_GROUP),
    ('encoding', ENCODING_GROUP),
    ('feature_eng', FEATURE_GROUP),
]


def build_combined_fn(positive_preds, method_registry):
    """
    Build a combined preprocessing function from positive-gain methods.
    Within each mutually exclusive group, pick the highest predicted gain.
    Apply in pipeline order.
    """
    selected = []
    for group_name, group_set in PIPELINE_GROUPS:
        candidates = [(m['Method'], m['Predicted_Gain'])
                      for m in positive_preds
                      if m['Method'] in group_set and m['Method'] in method_registry]
        if candidates:
            best_name = max(candidates, key=lambda x: x[1])[0]
            selected.append(best_name)

    if not selected:
        return None, []

    def combined(X_tr, X_va, y_tr):
        for name in selected:
            fn = method_registry[name]
            X_tr, X_va = fn(X_tr, X_va, y_tr)
        return X_tr, X_va

    return combined, selected


# ============================================================
# Run Validation
# ============================================================

print("=" * 75)
print("  GROUND-TRUTH VALIDATION ON TESTSET")
print("  Running actual 5-Fold CV to verify meta-model predictions")
print("=" * 75)

all_validation = []

for info in TEST_DATASETS:
    filepath = os.path.join(TESTSET_DIR, info["file"])
    target_col = info["target"]
    dataset_name = info["file"].replace(".csv", "")

    if not os.path.exists(filepath):
        print(f"\n--- {dataset_name} --- FILE NOT FOUND, skipping")
        continue

    print(f"\n{'=' * 70}")
    print(f"  {dataset_name}")
    print(f"{'=' * 70}")

    try:
        df = pd.read_csv(filepath)

        # Sample large datasets
        if len(df) > MAX_VAL_ROWS:
            print(f"  Sampling {MAX_VAL_ROWS} from {len(df)} rows for speed")
            df = df.sample(n=MAX_VAL_ROWS, random_state=42).reset_index(drop=True)

        # Separate X, y
        X = df.drop(columns=[target_col])
        y_raw = df[target_col]

        # Drop datetime columns
        dt_cols = X.select_dtypes(include=['datetime64', 'datetime64[ns]']).columns
        if len(dt_cols) > 0:
            X = X.drop(columns=dt_cols)

        # Encode target
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y_raw.astype(str)), name='target')

        print(f"  Shape: {X.shape[0]} x {X.shape[1]} | {y.nunique()} classes")

        # --- Get meta-model predictions ---
        mf = extract_meta_features(X, y)
        mf_df = pd.DataFrame([mf])
        feat_cols = joblib.load(f"{MODEL_DIR}/feature_cols.pkl")
        for c in feat_cols:
            if c not in mf_df.columns:
                mf_df[c] = 0

        predictions = []
        for fname in os.listdir(MODEL_DIR):
            if not fname.startswith('regressor_') or not fname.endswith('.pkl'):
                continue
            method = fname.replace('regressor_', '').replace('.pkl', '')
            mdl = joblib.load(os.path.join(MODEL_DIR, fname))
            if hasattr(mdl, 'feature_name_'):
                m_feats = mdl.feature_name_
                for c in m_feats:
                    if c not in mf_df.columns:
                        mf_df[c] = 0
                X_input = mf_df[m_feats]
            else:
                X_input = mf_df[feat_cols]
            pred = mdl.predict(X_input)[0]
            predictions.append({'Method': method, 'Predicted_Gain': pred})

        positive_preds = sorted(
            [p for p in predictions if p['Predicted_Gain'] > 0],
            key=lambda x: x['Predicted_Gain'], reverse=True
        )

        print(f"  Meta-model recommends {len(positive_preds)} methods with positive gain\n")

        # --- Baseline ---
        t0 = time.time()
        base_score, base_std = evaluate_method(X, y, method_fn=None)
        t_base = time.time() - t0

        if base_score is None:
            print(f"  Baseline FAILED, skipping dataset")
            continue

        print(f"  {'Method':<30s} {'Predicted':>10s} {'Actual':>10s} {'Match':>6s} {'Time':>7s}")
        print(f"  {'-' * 65}")
        print(f"  {'Baseline':<30s} {'---':>10s} {base_score:>10.4f} {'':>6s} {t_base:>6.1f}s")

        dataset_results = {
            'dataset': dataset_name,
            'baseline': base_score,
            'methods': {},
        }

        # --- Individual methods ---
        applicable = get_applicable_methods(X)

        for pred in positive_preds:
            method_name = pred['Method']
            predicted_gain = pred['Predicted_Gain']

            if method_name not in applicable:
                continue

            method_fn = applicable[method_name]
            t0 = time.time()
            score, std = evaluate_method(X, y, method_fn)
            elapsed = time.time() - t0

            if score is not None:
                actual_gain = score - base_score
                direction_match = (predicted_gain > 0) == (actual_gain > 0)
                mark = "\u2713" if direction_match else "\u2717"

                print(f"  {method_name:<30s} {predicted_gain:>+10.4f} {actual_gain:>+10.4f} {'  ' + mark:>6s} {elapsed:>6.1f}s")

                dataset_results['methods'][method_name] = {
                    'predicted_gain': predicted_gain,
                    'actual_gain': actual_gain,
                    'actual_score': score,
                    'direction_match': direction_match,
                }
            else:
                print(f"  {method_name:<30s} {predicted_gain:>+10.4f} {'FAILED':>10s} {'':>6s} {elapsed:>6.1f}s")

        # --- Combined pipeline ---
        combined_fn, selected_names = build_combined_fn(positive_preds, applicable)

        if combined_fn is not None and len(selected_names) > 1:
            t0 = time.time()
            combo_score, combo_std = evaluate_method(X, y, combined_fn)
            elapsed = time.time() - t0

            if combo_score is not None:
                combo_gain = combo_score - base_score
                print(f"  {'-' * 65}")
                print(f"  {'COMBINED PIPELINE':<30s} {'':>10s} {combo_gain:>+10.4f} {'':>6s} {elapsed:>6.1f}s")
                print(f"  Steps: {' -> '.join(selected_names)}")

                dataset_results['combined_gain'] = combo_gain
                dataset_results['combined_steps'] = selected_names

        all_validation.append(dataset_results)

    except Exception as e:
        print(f"  ERROR: {e}")
        continue

# ============================================================
# Final Summary
# ============================================================
if all_validation:
    print(f"\n\n{'=' * 75}")
    print("  VALIDATION SUMMARY")
    print(f"{'=' * 75}")
    print(f"  {'Dataset':<26s} {'Baseline':>9s} {'Best_Single':>13s} {'Combined':>10s} {'Direction':>10s}")
    print(f"  {'-' * 70}")

    total_correct = 0
    total_preds = 0

    for v in all_validation:
        ds = v['dataset']
        bl = v['baseline']

        # Best single method by actual gain
        if v['methods']:
            best = max(v['methods'].items(), key=lambda x: x[1]['actual_gain'])
            best_str = f"{best[1]['actual_gain']:+.4f}"
            for m_info in v['methods'].values():
                total_preds += 1
                if m_info['direction_match']:
                    total_correct += 1
        else:
            best_str = "N/A"

        combo_str = f"{v['combined_gain']:+.4f}" if 'combined_gain' in v else "N/A"

        ds_correct = sum(1 for m in v['methods'].values() if m['direction_match'])
        ds_total = len(v['methods'])
        dir_str = f"{ds_correct}/{ds_total}" if ds_total > 0 else "N/A"

        print(f"  {ds:<26s} {bl:>9.4f} {best_str:>13s} {combo_str:>10s} {dir_str:>10s}")

    print(f"  {'-' * 70}")
    if total_preds > 0:
        dir_acc = total_correct / total_preds * 100
        print(f"\n  Overall Direction Accuracy: {total_correct}/{total_preds} ({dir_acc:.1f}%)")
        print(f"  (meta-model correctly predicted whether a method helps or hurts)")
else:
    print("\nNo validation results.")

  GROUND-TRUTH VALIDATION ON TESTSET
  Running actual 5-Fold CV to verify meta-model predictions

  maternal_health_risk
  Shape: 1014 x 6 | 3 classes
  Meta-model recommends 2 methods with positive gain

  Method                          Predicted     Actual  Match    Time
  -----------------------------------------------------------------
  Baseline                              ---     0.8313           3.6s
  ClipOutliers                      +0.0207    -0.0039      ✗    3.5s
  ArithmeticInteractions            +0.0096    +0.0119      ✓    3.2s
  -----------------------------------------------------------------
  COMBINED PIPELINE                            +0.0158           3.2s
  Steps: ClipOutliers -> ArithmeticInteractions

  MIC
  Shape: 1699 x 111 | 8 classes
  Meta-model recommends 3 methods with positive gain

  Method                          Predicted     Actual  Match    Time
  -----------------------------------------------------------------
  Baseline                    

## Preview: Testset Datasets (First 10 Rows Each)

In [10]:
TESTSET_DIR = "testset"

TEST_FILES = [
    {"file": "maternal_health_risk.csv",      "target": "RiskLevel"},
    {"file": "MIC.csv",                       "target": "LET_IS"},
    {"file": "splice.csv",                    "target": "SiteType"},
    {"file": "hiva_agnostic.csv",             "target": "CompoundActivity"},
    {"file": "E-CommereShippingData.csv",     "target": "ArrivedLate"},
    {"file": "online_shoppers_intention.csv", "target": "Revenue"},
    {"file": "Amazon_employee_access.csv",    "target": "ResourceApproved"},
    {"file": "APSFailure.csv",                "target": "AirPressureSystemFailure"},
]

for info in TEST_FILES:
    filepath = os.path.join(TESTSET_DIR, info["file"])
    if not os.path.exists(filepath):
        print(f"--- {info['file']} --- NOT FOUND\n")
        continue

    df = pd.read_csv(filepath)
    name = info["file"].replace(".csv", "")
    target = info["target"]

    print(f"{'=' * 80}")
    print(f"  {name}  |  Shape: {df.shape}  |  Target: '{target}'")
    print(f"  Columns: {list(df.columns)}")
    print(f"{'=' * 80}")
    display(df.head(10))
    print()

  maternal_health_risk  |  Shape: (1014, 7)  |  Target: 'RiskLevel'
  Columns: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,55,120,90,6.80,98.0,66,mid risk
1,15,120,80,7.01,98.0,70,low risk
2,40,160,100,19.00,98.0,77,high risk
3,55,140,95,19.00,98.0,77,high risk
4,23,100,85,7.00,98.0,66,low risk
5,42,120,90,9.00,98.0,70,mid risk
6,30,120,80,6.80,98.0,70,mid risk
7,23,120,90,7.80,98.0,60,mid risk
8,17,90,60,11.00,101.0,78,high risk
9,25,120,80,7.80,98.0,66,low risk



  MIC  |  Shape: (1699, 112)  |  Target: 'LET_IS'
  Columns: ['AGE', 'SEX', 'INF_ANAM', 'STENOK_AN', 'FK_STENOK', 'IBS_POST', 'IBS_NASL', 'GB', 'SIM_GIPERT', 'DLIT_AG', 'ZSN_A', 'nr_11', 'nr_01', 'nr_02', 'nr_03', 'nr_04', 'nr_07', 'nr_08', 'np_01', 'np_04', 'np_05', 'np_07', 'np_08', 'np_09', 'np_10', 'endocr_01', 'endocr_02', 'endocr_03', 'zab_leg_01', 'zab_leg_02', 'zab_leg_03', 'zab_leg_04', 'zab_leg_06', 'S_AD_KBRIG', 'D_AD_KBRIG', 'S_AD_ORIT', 'D_AD_ORIT', 'O_L_POST', 'K_SH_POST', 'MP_TP_POST', 'SVT_POST', 'GT_POST', 'FIB_G_POST', 'ant_im', 'lat_im', 'inf_im', 'post_im', 'IM_PG_P', 'ritm_ecg_p_01', 'ritm_ecg_p_02', 'ritm_ecg_p_04', 'ritm_ecg_p_06', 'ritm_ecg_p_07', 'ritm_ecg_p_08', 'n_r_ecg_p_01', 'n_r_ecg_p_02', 'n_r_ecg_p_03', 'n_r_ecg_p_04', 'n_r_ecg_p_05', 'n_r_ecg_p_06', 'n_r_ecg_p_08', 'n_r_ecg_p_09', 'n_r_ecg_p_10', 'n_p_ecg_p_01', 'n_p_ecg_p_03', 'n_p_ecg_p_04', 'n_p_ecg_p_05', 'n_p_ecg_p_06', 'n_p_ecg_p_07', 'n_p_ecg_p_08', 'n_p_ecg_p_09', 'n_p_ecg_p_10', 'n_p_ecg_p_11'

,AGE,SEX,INF_ANAM,STENOK_AN,FK_STENOK,IBS_POST,IBS_NASL,GB,SIM_GIPERT,DLIT_AG,...,NOT_NA_2_n,NOT_NA_3_n,LID_S_n,B_BLOK_S_n,ANT_CA_S_n,GEPAR_S_n,ASP_S_n,TIKL_S_n,TRENT_S_n,LET_IS
0,60.0,1,0.0,0.0,0.0,0.0,NaN,2.0,0.0,NaN,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,alive
1,65.0,1,2.0,5.0,2.0,2.0,NaN,2.0,0.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,alive
2,65.0,0,0.0,0.0,0.0,0.0,NaN,2.0,0.0,6.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,alive
3,55.0,1,1.0,1.0,2.0,1.0,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,alive
4,69.0,0,0.0,6.0,2.0,1.0,NaN,2.0,0.0,7.0,...,0.0,2.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,alive
5,76.0,1,0.0,3.0,2.0,2.0,NaN,3.0,0.0,7.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,alive
6,62.0,1,3.0,6.0,3.0,2.0,NaN,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,alive
7,61.0,1,1.0,6.0,2.0,1.0,NaN,2.0,0.0,6.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,alive
8,42.0,1,0.0,0.0,0.0,2.0,NaN,2.0,0.0,2.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,alive
9,57.0,0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,...,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,cardiogenic_shock



  splice  |  Shape: (3190, 61)  |  Target: 'SiteType'
  Columns: ['position_-30', 'position_-29', 'position_-28', 'position_-27', 'position_-26', 'position_-25', 'position_-24', 'position_-23', 'position_-22', 'position_-21', 'position_-20', 'position_-19', 'position_-18', 'position_-17', 'position_-16', 'position_-15', 'position_-14', 'position_-13', 'position_-12', 'position_-11', 'position_-10', 'position_-9', 'position_-8', 'position_-7', 'position_-6', 'position_-5', 'position_-4', 'position_-3', 'position_-2', 'position_-1', 'position_1', 'position_2', 'position_3', 'position_4', 'position_5', 'position_6', 'position_7', 'position_8', 'position_9', 'position_10', 'position_11', 'position_12', 'position_13', 'position_14', 'position_15', 'position_16', 'position_17', 'position_18', 'position_19', 'position_20', 'position_21', 'position_22', 'position_23', 'position_24', 'position_25', 'position_26', 'position_27', 'position_28', 'position_29', 'position_30', 'SiteType']


...

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.000000,0,0.00,6,1156.500000,0.000000,0.033333,0.000000,0.0,Nov,2,2,1,20,Returning_Visitor,False,False
1,4,52.000000,1,7.00,46,3087.000000,0.003774,0.021384,16.946438,0.0,Mar,2,2,3,8,Returning_Visitor,False,True
2,4,106.500000,0,0.00,12,806.250000,0.012500,0.029167,0.000000,0.0,Dec,2,2,1,2,Returning_Visitor,False,False
3,9,497.166667,0,0.00,20,1170.166667,0.012121,0.023485,0.000000,0.0,Nov,3,2,3,2,Returning_Visitor,False,False
4,0,0.000000,0,0.00,3,17.000000,0.000000,0.033333,0.000000,0.0,Mar,1,1,1,3,Returning_Visitor,False,False
5,0,0.000000,0,0.00,12,363.000000,0.016667,0.047222,0.000000,0.0,Nov,4,1,4,2,Returning_Visitor,False,False
6,1,11.500000,1,94.00,98,6212.779517,0.024163,0.033992,0.000000,0.0,Nov,3,2,4,10,Returning_Visitor,True,False
7,9,174.208333,2,383.75,151,9018.123741,0.012821,0.024593,0.000000,0.0,Nov,1,2,1,1,Returning_Visitor,True,False
8,0,0.000000,0,0.00,9,165.000000,0.000000,0.022222,0.000000,1.0,May,2,2,4,4,Returning_Visitor,True,False
9,0,0.000000,0,0.00,6,118.333333,0.033333,0.044444,0.000000,0.0,Nov,3,2,1,3,Returning_Visitor,True,False



  Amazon_employee_access  |  Shape: (32769, 10)  |  Target: 'ResourceApproved'
  Columns: ['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE', 'ResourceApproved']


,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE,ResourceApproved
0,37793,81744,117902,117903,118783,118451,130134,118453,118454,Yes
1,40309,1541,117961,118225,123173,119093,123174,119095,119096,Yes
2,27356,205,117961,118386,118746,118784,147114,290919,118786,Yes
3,5173,8229,117961,118300,121305,119351,149246,3130,119353,Yes
4,77207,51791,117961,119256,120943,118995,280788,292795,118997,Yes
5,82641,196,117961,118413,121639,118451,130134,118453,118454,Yes
6,43294,4566,119062,119091,118992,118028,130134,117887,118030,No
7,34924,3838,117961,118225,119924,118321,117906,290919,118322,Yes
8,30830,16803,119596,119597,120823,118890,123957,118398,118892,Yes
9,80175,23136,117961,118052,119742,117905,117906,290919,117908,Yes



  APSFailure  |  Shape: (76000, 171)  |  Target: 'AirPressureSystemFailure'
  Columns: ['aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ag_000', 'ag_001', 'ag_002', 'ag_003', 'ag_004', 'ag_005', 'ag_006', 'ag_007', 'ag_008', 'ag_009', 'ah_000', 'ai_000', 'aj_000', 'ak_000', 'al_000', 'am_0', 'an_000', 'ao_000', 'ap_000', 'aq_000', 'ar_000', 'as_000', 'at_000', 'au_000', 'av_000', 'ax_000', 'ay_000', 'ay_001', 'ay_002', 'ay_003', 'ay_004', 'ay_005', 'ay_006', 'ay_007', 'ay_008', 'ay_009', 'az_000', 'az_001', 'az_002', 'az_003', 'az_004', 'az_005', 'az_006', 'az_007', 'az_008', 'az_009', 'ba_000', 'ba_001', 'ba_002', 'ba_003', 'ba_004', 'ba_005', 'ba_006', 'ba_007', 'ba_008', 'ba_009', 'bb_000', 'bc_000', 'bd_000', 'be_000', 'bf_000', 'bg_000', 'bh_000', 'bi_000', 'bj_000', 'bk_000', 'bl_000', 'bm_000', 'bn_000', 'bo_000', 'bp_000', 'bq_000', 'br_000', 'bs_000', 'bt_000', 'bu_000', 'bv_000', 'bx_000', 'by_000', 'bz_000', 'ca_000', 'cb_000', 'cc_000', 'cd_000', 'ce_000', 'cf

,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,ag_003,...,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000,AirPressureSystemFailure
0,4090,0.0,2.680000e+02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,neg
1,12,0.0,8.000000e+00,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,38.0,32.0,60.0,76.0,0.0,0.0,0.0,0.0,neg
2,378,NaN,3.200000e+01,32.0,0.0,0.0,0.0,0.0,0.0,0.0,...,346.0,386.0,14064.0,22.0,30.0,0.0,0.0,0.0,0.0,neg
3,10616,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,6882.0,...,90948.0,14080.0,2914.0,814.0,422.0,268.0,0.0,0.0,0.0,neg
4,38218,NaN,4.140000e+02,408.0,0.0,0.0,0.0,0.0,0.0,13824.0,...,193228.0,364644.0,498410.0,196434.0,43556.0,42866.0,854.0,0.0,0.0,neg
5,562,NaN,3.600000e+01,36.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1242.0,1540.0,2442.0,7706.0,6554.0,26.0,0.0,0.0,0.0,neg
6,49796,NaN,5.000000e+02,444.0,0.0,0.0,0.0,0.0,0.0,0.0,...,232770.0,451794.0,358514.0,259946.0,171928.0,270312.0,12488.0,0.0,0.0,neg
7,38288,NaN,1.260000e+02,100.0,0.0,0.0,0.0,0.0,0.0,0.0,...,172166.0,396108.0,425744.0,228434.0,119178.0,104240.0,734.0,0.0,0.0,neg
8,2742,NaN,2.000000e+02,176.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6330.0,21718.0,72162.0,7352.0,1840.0,1228.0,0.0,0.0,0.0,neg
9,414,0.0,2.130706e+09,6.0,8.0,0.0,0.0,0.0,0.0,0.0,...,1550.0,2536.0,3386.0,6636.0,1362.0,0.0,0.0,0.0,0.0,neg
